<a href="https://colab.research.google.com/github/rollie90/Bank_Churn_Analysis/blob/main/Churn_Analysis_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import duckdb
import numpy as np
import pandas as pd

path = '/content/Bank Customer Churn Prediction.csv'
query = f"""
SELECT * FROM read_csv_auto('{path}')
"""

cla = duckdb.query(query).to_df() #For Churn Level Analysis

print(f"Dataset sucessfully loaded. Total registers: {len(cla)}")
print(cla.head())

Dataset sucessfully loaded. Total registers: 10000
   customer_id  credit_score country  gender  age  tenure    balance  \
0     15634602           619  France  Female   42       2       0.00   
1     15647311           608   Spain  Female   41       1   83807.86   
2     15619304           502  France  Female   42       8  159660.80   
3     15701354           699  France  Female   39       1       0.00   
4     15737888           850   Spain  Female   43       2  125510.82   

   products_number  credit_card  active_member  estimated_salary  churn  
0                1            1              1         101348.88      1  
1                1            0              1         112542.58      0  
2                3            1              0         113931.57      1  
3                2            0              0          93826.63      0  
4                1            1              1          79084.10      0  


In [ ]:
cla.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       10000 non-null  int64  
 1   credit_score      10000 non-null  int64  
 2   country           10000 non-null  object 
 3   gender            10000 non-null  object 
 4   age               10000 non-null  int64  
 5   tenure            10000 non-null  int64  
 6   balance           10000 non-null  float64
 7   products_number   10000 non-null  int64  
 8   credit_card       10000 non-null  int64  
 9   active_member     10000 non-null  int64  
 10  estimated_salary  10000 non-null  float64
 11  churn             10000 non-null  int64  
dtypes: float64(2), int64(8), object(2)
memory usage: 937.6+ KB


In [ ]:
cla.describe(include='all')

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
count,1.000000e+04,10000.000000,10000,10000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
unique,NaN,NaN,3,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,France,Male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,5014,5457,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,1.569094e+07,650.528800,NaN,NaN,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,7.193619e+04,96.653299,NaN,NaN,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.556570e+07,350.000000,NaN,NaN,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,1.562853e+07,584.000000,NaN,NaN,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,1.569074e+07,652.000000,NaN,NaN,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,1.575323e+07,718.000000,NaN,NaN,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000


In [ ]:
query_kpi = f"""
SELECT
    country,
    COUNT(*) as total_clients,
    SUM(churn) as total_churn,
    ROUND(AVG(churn) * 100, 2) as churn_percentage
FROM read_csv_auto('{path}')
GROUP BY country
ORDER BY churn_percentage DESC
"""

kpi_country = duckdb.query(query_kpi).to_df()  #Churn Percetage by country
print(kpi_country)


   country  total_clients  total_churn  churn_percentage
0  Germany           2509        814.0             32.44
1    Spain           2477        413.0             16.67
2   France           5014        810.0             16.15


In [ ]:
query_kpi = f"""
SELECT
    country,
    SUM(active_member) as total_active_members,
    ROUND(AVG(active_member) * 100, 2) as percentage_active_member
FROM read_csv_auto('{path}')
GROUP BY country
ORDER BY percentage_active_member DESC
"""

kpi_active = duckdb.query(query_kpi).to_df()  #Active Members Percentage
print(kpi_active)

   country  total_active_members  percentage_active_member
0    Spain                1312.0                     52.97
1   France                2591.0                     51.68
2  Germany                1248.0                     49.74


In [ ]:
query_kpi = f"""
SELECT
    country,
    products_number,
    ROUND(AVG(churn) * 100,2) as churn_percentage_germany
FROM read_csv_auto('{path}')
WHERE country = 'Germany'
GROUP BY country, products_number
ORDER BY churn_percentage_germany DESC
"""

kpi_germany_products = duckdb.query(query_kpi).to_df()  #Churn Percentage in Germany
print(kpi_germany_products)

   country  products_number  churn_percentage_germany
0  Germany                4                    100.00
1  Germany                3                     89.58
2  Germany                1                     42.85
3  Germany                2                     12.12


In [ ]:
query_kpi = f"""
SELECT
    country,
    products_number,
    ROUND(AVG(churn) * 100,2) as general_churn_percentage
FROM read_csv_auto('{path}')
GROUP BY country, products_number
ORDER BY general_churn_percentage DESC
"""

kpi_germany_products = duckdb.query(query_kpi).to_df() #Churn Percentage by Number of Owned Bank Products
print(kpi_germany_products)

    country  products_number  general_churn_percentage
0     Spain                4                    100.00
1   Germany                4                    100.00
2    France                4                    100.00
3   Germany                3                     89.58
4    France                3                     78.85
5     Spain                3                     78.79
6   Germany                1                     42.85
7    France                1                     22.43
8     Spain                1                     21.87
9   Germany                2                     12.12
10    Spain                2                      7.35
11   France                2                      5.70


In [ ]:
query_kpi = f"""
SELECT
    country,
    tenure,
    ROUND(AVG(churn) * 100, 2) as tenure_churn_percentage
FROM read_csv_auto('{path}')
GROUP BY country, tenure
ORDER BY tenure_churn_percentage DESC
"""

kpi_churn_tenure = duckdb.query(query_kpi).to_df()  #Churn Percentage by Tenure All Countries
print(kpi_churn_tenure)

    country  tenure  tenure_churn_percentage
0   Germany       1                    39.77
1   Germany       5                    34.75
2   Germany       9                    34.57
3   Germany       0                    34.29
4   Germany       6                    33.48
5   Germany       8                    32.95
6   Germany      10                    32.81
7   Germany       4                    32.76
8   Germany       3                    32.18
9   Germany       7                    27.11
10    Spain       0                    23.30
11  Germany       2                    23.27
12   France      10                    19.75
13    Spain       2                    18.95
14    Spain       1                    18.18
15    Spain       6                    18.14
16    Spain       3                    17.90
17   France       9                    17.46
18   France       2                    17.14
19    Spain       4                    17.14
20   France       0                    17.07
21   Franc

In [ ]:
query_kpi = f"""
SELECT
    country,
    tenure,
    ROUND(AVG(churn) * 100, 2) as tenure_germany_churn_percentage
FROM read_csv_auto('{path}')
WHERE country = 'Germany'
GROUP BY country, tenure
ORDER BY tenure_germany_churn_percentage DESC
"""

ten_churn_ger = duckdb.query(query_kpi).to_df()  #Churn Percentage by Tenure Germany
print(ten_churn_ger)

    country  tenure  tenure_germany_churn_percentage
0   Germany       1                            39.77
1   Germany       5                            34.75
2   Germany       9                            34.57
3   Germany       0                            34.29
4   Germany       6                            33.48
5   Germany       8                            32.95
6   Germany      10                            32.81
7   Germany       4                            32.76
8   Germany       3                            32.18
9   Germany       7                            27.11
10  Germany       2                            23.27


In [ ]:
query_kpi = f"""
SELECT
    country,
    churn,
    COUNT(*) as total_clients,
    ROUND(AVG(tenure),2) as avg_tenure_germany
FROM read_csv_auto('{path}')
WHERE country = 'Germany'
GROUP BY country, churn
ORDER BY churn DESC
"""

kpi_tenure_germany = duckdb.query(query_kpi).to_df() #Average Tenure in Germany by Churn Status
print(kpi_tenure_germany)

   country  churn  total_clients  avg_tenure_germany
0  Germany      1            814                5.01
1  Germany      0           1695                5.01


In [ ]:
query_kpi = f"""
SELECT
    country,
    ROUND(AVG(tenure), 2) as avg_tenure_4_prod
FROM read_csv_auto('{path}')
WHERE products_number = 4
GROUP BY country
ORDER BY avg_tenure_4_prod
"""
kpi_churn_4 = duckdb.query(query_kpi).to_df() #Average Tenure Of Clients Who Own 4 Bank Products
print(kpi_churn_4)

   country  avg_tenure_4_prod
0  Germany               4.63
1    Spain               4.86
2   France               5.97


In [ ]:
query_kpi = f"""
SELECT
    country,
    churn,
    ROUND(AVG(balance),2) as avg_balance_country
FROM read_csv_auto('{path}')
WHERE churn = 0
GROUP BY country, churn
ORDER BY avg_balance_country DESC
"""

kpi_balance_country = duckdb.query(query_kpi).to_df() #Churn AVG Account Balance
print(kpi_balance_country)

   country  churn  avg_balance_country
0  Germany      0            119427.11
1   France      0             60339.28
2    Spain      0             59678.07


In [ ]:
active_balance_df = cla[cla['balance'] > 0]
churned_customers = active_balance_df[active_balance_df['churn']==1]
retained_customers = active_balance_df[active_balance_df['churn']==0]


total_loss = churned_customers['balance'].sum()
total_bank_capital = active_balance_df['balance'].sum()
avg_churn_balance = churned_customers['balance'].mean()
avg_retained_balance = retained_customers['balance'].mean()
total_relative_change = ((avg_churn_balance-avg_retained_balance)/avg_retained_balance)*100
drain_percentage = (total_loss/total_bank_capital)*100

print(f"Total Capital Lost (Liquidity Drain): ${total_loss:,.2f}")
print(f"Average Balance of Exited Customers: ${avg_churn_balance:,.2f}")
print(f"Average Balance of Retained Customers: ${avg_retained_balance:,.2f}")
print(f"Value Differential (Churn vs. Retained): {total_relative_change:,.2f}%")
print(f"Total Bank Capital Drain Percentage: {drain_percentage:,.2f}%")


Total Capital Lost (Liquidity Drain): $185,588,094.63
Average Balance of Exited Customers: $120,746.97
Average Balance of Retained Customers: $119,535.86
Value Differential (Churn vs. Retained): 1.01%
Total Bank Capital Drain Percentage: 24.26%


In [ ]:
bank_germany = active_balance_df[active_balance_df['country']=='Germany']
churned_germany = churned_customers[churned_customers['country']=='Germany']
retained_germany = retained_customers[retained_customers['country']=='Germany']

total_bank_capital_germany = bank_germany['balance'].sum()
churn_germany_balance = churned_germany['balance'].sum()
retained_germany_balance = retained_germany['balance'].sum()
drain_percentage_germany = (churn_germany_balance/total_bank_capital_germany)*100
avg_churn_germany = churned_germany['balance'].mean()
avg_retained_germany = retained_germany['balance'].mean()
relative_change_germany = ((avg_churn_germany-avg_retained_germany)/avg_retained_germany)*100

print(f"Total Capital in Germany: ${total_bank_capital_germany:,.2f}")
print(f"Total Balance of Exited Customers in Germany: ${churn_germany_balance:,.2f}")
print(f"Total Balance of Retained Customers in Germany: ${retained_germany_balance:,.2f}")
print(f"Total Bank Capital Drain Percentage in Germany: {drain_percentage_germany:,.2f}%")
print(f"Average Balance of Exited Customers in Germany: ${avg_churn_germany:,.2f}")
print(f"Average Balance of Retained Customers in Germany: ${avg_retained_germany:,.2f}")
print(f"Value Differential (Churn vs. Retained) in Germany: {relative_change_germany:,.2f}%")


Total Capital in Germany: $300,402,861.38
Total Balance of Exited Customers in Germany: $97,973,915.53
Total Balance of Retained Customers in Germany: $202,428,945.85
Total Bank Capital Drain Percentage in Germany: 32.61%
Average Balance of Exited Customers in Germany: $120,361.08
Average Balance of Retained Customers in Germany: $119,427.11
Value Differential (Churn vs. Retained) in Germany: 0.78%


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

germany = cla[cla['country'] == 'Germany']

plt.figure(figsize=(10, 6))

sns.histplot(data=germany, x='age', hue='churn', bins=20, kde=True, palette='viridis')

plt.title('Churn Age Distribution in Germany')
plt.xlabel('Age')
plt.ylabel('Client Number')
plt.show()



In [ ]:
germany_df = cla[cla['country'] == 'Germany'].copy()

chosen_columns = germany_df.select_dtypes(include=[np.number]).drop(columns=['customer_id'], errors='ignore')

correlation = chosen_columns.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(x='churn', y='balance', data=germany_df, palette='Set2', hue='churn', showfliers=False)

sns.stripplot(x='churn', y='balance', data=germany_df, color='black', alpha=0.1, size=2)

plt.title('Balance Comparison: Remaining Clients vs. Churn (Germany)', fontsize=14)
plt.xlabel('Churn (0=Stays, 1=Leaves)', fontsize=12)
plt.ylabel('Account Balance', fontsize=12)
plt.show()



In [ ]:
query_kpi = f"""
SELECT
credit_card,
active_member,
ROUND(AVG(churn) * 100,2) as churn_percentage,
COUNT(*) as client_number
FROM read_csv_auto('{path}')
GROUP BY credit_card, active_member
ORDER BY churn_percentage DESC
"""

print(duckdb.query(query_kpi).to_df())

df_germany = cla[cla['country'] == 'Germany']

plt.figure(figsize=(10, 6))
sns.barplot(data=df_germany, x='credit_card', y='churn', hue='active_member', palette='magma')

print("\n" * 2)
plt.axhline(y=0.5, color='red', linestyle='--', label='Limit 50%') # Reference Line
plt.title('Churn Probability in Germany: Credit Card vs Activity')
plt.ylabel('Churn Percentage (Proportion)')
plt.xlabel('¿Owns a Credit Card? (0=No, 1=Yes)')
plt.legend(title='Active Member (0=No, 1=Yes)')
plt.show()

